# Avaliação do Modelo de Risco de Crédito

Este notebook lê os artefatos produzidos por `Model/train.py` e traduz resultados técnicos para decisão de crédito. O holdout de 20% é consultado uma única vez após seleção e tuning do vencedor.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from MLOps import storage

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Metodologia

1. Split estratificado 80/20.
2. CV-AUC em 5 folds para os três candidatos.
3. GridSearchCV somente no vencedor.
4. Fit final no treino completo.
5. Avaliação única no holdout: AUC-ROC, KS, recall, precision, F1 e matriz de confusão.

In [ ]:
metrics=storage.read_json("Model/metrics.json")
metrics

## 2. Comparação dos algoritmos

In [ ]:
comparison=storage.read_csv("reports/model_comparison.csv")
display(comparison)
comparison.set_index('model')['cv_auc_mean'].plot(kind='bar',ylim=(.5,1),title='CV-AUC média por algoritmo'); plt.ylabel('AUC'); plt.show()

## 3. Resultado final do vencedor

In [ ]:
hold=metrics['holdout']
summary=pd.Series({
 'Modelo vencedor':metrics['best_model'],
 'AUC-ROC':hold['auc_roc'],
 'KS':hold['ks'],
 'Recall inadimplente':hold['recall_default'],
 'Precision inadimplente':hold['precision_default'],
 'F1 inadimplente':hold['f1_default'],
 'Acurácia':hold['accuracy'],
 'Threshold':hold['threshold']
})
display(summary.to_frame('valor'))

## 4. Curva ROC

In [ ]:
roc=storage.read_csv("reports/roc_curve_best_model.csv")
plt.plot(roc.fpr,roc.tpr,label=f"AUC={hold['auc_roc']:.3f}")
plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('Curva ROC — holdout'); plt.legend(); plt.show()

## 5. Matriz de confusão

- **VN:** bom pagador aprovado corretamente.
- **FP:** bom pagador revisado/negado; perda potencial de receita.
- **FN:** inadimplente aprovado; erro financeiro mais caro.
- **VP:** inadimplente corretamente identificado.

In [ ]:
cm=hold['confusion_matrix']
mat=np.array([[cm['true_negative'],cm['false_positive']],[cm['false_negative'],cm['true_positive']]])
fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(mat)
for i in range(2):
    for j in range(2): ax.text(j,i,f'{mat[i,j]:,}',ha='center',va='center',fontsize=14)
ax.set_xticks([0,1],['Predito bom','Predito risco']); ax.set_yticks([0,1],['Real bom','Real inadimplente']); ax.set_title(f"Matriz de confusão — threshold {hold['threshold']:.2f}"); plt.colorbar(im); plt.show()

## 6. Trade-off de threshold

In [ ]:
thresholds=storage.read_csv("reports/threshold_analysis.csv")
display(thresholds)
thresholds.set_index('threshold')[['approval_rate','recall_default','precision_default','false_negative_rate']].plot(marker='o',title='Impacto do threshold'); plt.ylim(0,1); plt.show()

## 7. Importância e explicabilidade

In [ ]:
if storage.exists("reports/feature_importance.csv"):
    native=storage.read_csv("reports/feature_importance.csv").head(20)
    display(native)
    native.sort_values('abs_value').plot.barh(x='feature',y='abs_value',legend=False,title='Importância nativa — top 20'); plt.show()

In [ ]:
if storage.exists("reports/permutation_importance.csv"):
    perm=storage.read_csv("reports/permutation_importance.csv").head(20)
    display(perm)
    perm.sort_values('importance_mean').plot.barh(x='feature',y='importance_mean',legend=False,title='Permutation importance — top 20'); plt.show()

In [ ]:
if storage.exists("reports/shap_importance.csv"):
    shap_imp=storage.read_csv("reports/shap_importance.csv").head(20)
    display(shap_imp)
    shap_imp.sort_values('mean_abs_shap').plot.barh(x='feature',y='mean_abs_shap',legend=False,title='SHAP global — top 20'); plt.show()
else:
    print(storage.read_json("reports/shap_status.json") if storage.exists("reports/shap_status.json") else 'SHAP não executado.')

## 8. Interpretação de negócio

- AUC mede capacidade de ordenar solicitações por risco.
- KS mede separação entre bons e maus pagadores.
- Recall indica quantos inadimplentes reais são capturados.
- Threshold menor aumenta proteção contra inadimplência, porém envia mais bons clientes para revisão.
- O modelo entrega probabilidade; a política de crédito decide o threshold e as ações para cada faixa.
- Casos próximos ao limite devem ser revisados por analista, e não decididos automaticamente.